# SME Retail Revenue Intelligence Platform - New Improved Version

Notebook นี้เลือกโจทย์ **SME retail** เพราะเป็นธุรกิจที่เห็น pain point เรื่องรายได้ชัดมาก: ของที่ขายดีมักหมดก่อนเติม, ของบางกลุ่มค้างสต็อกหรือหมดอายุ, และโปรโมชันบางอันลดราคาแล้วไม่ได้ช่วยกำไรจริง ๆ

แนวคิดหลักคือทำ Data Science solution ที่ไม่ได้จบแค่การ forecast แต่แปลงผลเป็น **action list ที่เจ้าของร้านหรือผู้จัดการสาขาใช้ตัดสินใจได้ทันที** เช่น สาขาไหนควรเติมสินค้าอะไร จำนวนเท่าไร มีโอกาสรายได้เท่าไร และโปรโมชันไหนควรทำต่อหรือควรปรับ

## สิ่งที่ solution นี้ทำ

1. สร้าง mock data ให้ครบ 8 ตารางตามโจทย์: sales, PO, product, promotion, store, customer, warehouse และ stock movement
2. ตรวจคุณภาพข้อมูล เช่น missing value, duplicate, negative qty/price และ date logic
3. ทำ EDA เพื่อดู revenue, demand, category, customer segment และ expiry risk
4. สร้าง feature สำหรับ demand forecasting เช่น calendar, promotion, lag sales และ rolling sales
5. เทียบ baseline กับ ML model แล้วเลือก model ที่ดีที่สุดแบบไม่ claim เกินจริง
6. แปลง forecast เป็น reorder recommendation พร้อม revenue opportunity และ stockout risk
7. วิเคราะห์ promotion uplift และ ROI เพื่อแนะนำว่าโปรไหนควรทำต่อ
8. วาง MLOps concept: model registry, monitoring, retraining trigger และ saved model artifact


## แผนการทำงาน 1 สัปดาห์

| วัน | งานหลัก | Output ที่ได้ |
|---|---|---|
| Day 1 | ทำความเข้าใจโจทย์, นิยาม KPI, ออกแบบ schema | Problem framing + data dictionary |
| Day 2 | สร้าง/เตรียม mock data 8 ตาราง และตรวจ data quality | Raw CSV + validation report |
| Day 3 | ทำ EDA และ feature engineering | Insight เบื้องต้น + feature table |
| Day 4 | สร้าง baseline และ train ML model | Model metrics: WAPE, MAE, RMSE |
| Day 5 | ทำ inventory recommendation และ promotion analytics | Reorder CSV + promotion recommendation |
| Day 6 | วาง MLOps, monitoring, model registry และ saved artifacts | Model artifact + registry + schema |
| Day 7 | สรุปผลสำหรับลูกค้าและเตรียม package ส่งมอบ | Executive summary + deliverables |


## ทำไมเลือก ML/AI solution นี้

โจทย์คือ maximize revenue ให้ SME retail ดังนั้น model ที่เหมาะที่สุดไม่ใช่ model ที่ซับซ้อนที่สุด แต่เป็น model ที่ช่วยตอบคำถามเชิงธุรกิจได้เร็วและอธิบายต่อได้

ใน notebook นี้ใช้แนวทาง **baseline vs ML model comparison**:

- Baseline: 14-day rolling average ใช้เป็นตัวเทียบง่าย ๆ ว่าถ้าไม่ใช้ ML จะทำได้แค่ไหน
- ML candidates: Gradient Boosting และ Random Forest เพราะเหมาะกับ tabular retail data, รับ feature หลายแบบได้ดี และอธิบาย feature importance ได้
- Metric หลัก: WAPE เพราะอ่านง่ายในมุมธุรกิจ retail ว่าคลาดเคลื่อนจาก demand จริงกี่เปอร์เซ็นต์โดยถ่วงตามปริมาณยอดขาย

ถ้า ML ไม่ชนะ baseline จะไม่เรียกว่า production model ทันที แต่จะตั้งเป็น candidate ที่ต้องปรับปรุงต่อ นี่เป็น logic ที่ตั้งใจใส่ไว้เพื่อให้การสรุปผลไม่ overclaim


## Mock columns ที่เพิ่มจากโจทย์ และเหตุผลที่ใช้

โจทย์ให้ schema หลักไว้แล้ว แต่ในงานจริง retail ต้องมี context เพิ่มเพื่อให้ model และ business recommendation มีประโยชน์ขึ้น จึงเพิ่ม columns บางตัวดังนี้

| Table | Column ที่เพิ่ม | ใช้ทำอะไร |
|---|---|---|
| product_master | shelf_life_days | ใช้ดู expiry risk โดยเฉพาะ Fresh Food และสินค้าที่หมดอายุเร็ว |
| product_master | gross_margin_pct | ใช้คำนวณ margin และดูว่า promotion คุ้มจริงไหม |
| promotion_master | promotion_name | ทำให้วิเคราะห์โปรโมชันเป็น campaign ได้ อ่านผลเข้าใจง่ายขึ้น |
| store_master | province | ช่วยสื่อสาร recommendation ให้คนธุรกิจเข้าใจว่าเป็นสาขา/จังหวัดไหน |
| store_master | store_size_sqm | ใช้เป็น context ของศักยภาพสาขาในอนาคต |
| store_master | opened_year | ใช้เป็น context เรื่อง maturity ของสาขา |
| warehouse_master | province | ใช้ดู logistics node และอธิบาย flow คลัง-สาขา |
| warehouse_master | capacity_units | ใช้เป็น context สำหรับ supply capacity ในอนาคต |

การเพิ่ม column เหล่านี้ไม่ได้เพิ่มเพื่อให้ข้อมูลดูเยอะ แต่เพิ่มเพื่อให้คำแนะนำสุดท้ายตอบ business ได้ดีขึ้น เช่น เติม stock เท่าไร, เสี่ยงหมดอายุไหม, โปรโมชันคุ้ม margin หรือเปล่า


## MLOps concept และการใช้ AI ช่วยทำงาน

MLOps สำหรับโปรเจกต์นี้ออกแบบให้เหมาะกับงาน 1 สัปดาห์ ไม่ได้ทำ production เต็มรูปแบบ แต่แสดงให้เห็นว่าถ้าลูกค้าจะต่อยอดควรเดินอย่างไร

แนวคิด MLOps ที่ใส่ไว้:

- เก็บ raw CSV ทั้ง 8 ตาราง เพื่อย้อนกลับไปตรวจข้อมูลได้
- บันทึก feature schema เพื่อกันปัญหา column หายหรือ format เปลี่ยนตอน inference
- save trained model เป็น `.joblib`
- ทำ model registry เก็บ train period, test period, metric, status และ retraining policy
- monitoring WAPE/drift รายสัปดาห์ และ trigger retraining เมื่อ error หรือ drift สูงเกิน threshold

AI ถูกใช้เป็นผู้ช่วยใน workflow เช่น ช่วย review idea, ช่วยตรวจว่า schema ตรงโจทย์ไหม, ช่วยสรุป insight ให้เป็นภาษาคนธุรกิจ และช่วย draft executive summary แต่การตัดสินใจหลักยังอิงจาก metric และ business logic ใน notebook


## Output ที่ลูกค้าจะได้รับ

ลูกค้าจะไม่ได้รับแค่ notebook แต่จะได้รับ package ที่นำไปคุยต่อหรือทำ dashboard ได้ทันที:

1. Mock dataset 8 ตารางในรูปแบบ CSV
2. EDA charts สำหรับดู revenue, demand, category, customer segment และ promotion performance
3. Demand forecast model พร้อม metric report
4. `model_recommendation_output.csv` สำหรับบอกว่าสาขาไหนควรเติมสินค้าอะไร จำนวนเท่าไร
5. Promotion ROI/uplift analysis เพื่อดูว่าโปรไหนควรทำต่อหรือปรับ
6. Model artifact และ feature schema สำหรับต่อยอด inference
7. MLOps model registry และ monitoring concept
8. Executive summary ที่สรุปเป็น action สำหรับเจ้าของ SME retail


## 0. 📦 Import Libraries & Configuration

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
import json
from pathlib import Path
import joblib

# ML
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)

# Color palette
COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'success':   '#C73E1D',
    'neutral':   '#3B1F2B',
    'pastel':    ['#AED6F1', '#A9DFBF', '#FAD7A0', '#F1948A', '#D7BDE2']
}

print("✅ All libraries imported successfully")
print(f"📅 Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


---
## 1. 🏗️ Mock Data Generation
**8 tables** designed to cover the full retail supply-chain schema.  
Data covers **2025-01-01 → 2025-12-31** (full year, 3 stores, 50 products, 500 customers).


In [ ]:

# ── Master dimensions ─────────────────────────────────────────────────────────
np.random.seed(42)

N_PRODUCTS   = 50
N_CUSTOMERS  = 500
N_STORES     = 3
N_WAREHOUSES = 2
START_DATE   = pd.Timestamp('2025-01-01')
END_DATE     = pd.Timestamp('2025-12-31')

# 1. product_master
categories   = ['Beverage', 'Snack', 'Fresh Food', 'Personal Care', 'Household']
shelf_map    = {'Beverage':180, 'Snack':120, 'Fresh Food':7, 'Personal Care':365, 'Household':730}
margin_map   = {'Beverage':0.35, 'Snack':0.32, 'Fresh Food':0.20, 'Personal Care':0.45, 'Household':0.38}

product_cats  = np.random.choice(categories, N_PRODUCTS)
product_master = pd.DataFrame({
    'product_id':      [f'P{str(i+1).zfill(4)}' for i in range(N_PRODUCTS)],
    'product_taxonomies': product_cats,
    'price':           np.round(np.random.uniform(15, 120, N_PRODUCTS), 2),
    'shelf_life_days': [shelf_map[c] + np.random.randint(-10, 10) for c in product_cats],
    'gross_margin_pct': [round(margin_map[c] + np.random.uniform(-0.03, 0.03), 3) for c in product_cats]
})
print(f"✅ product_master: {product_master.shape}")

# 2. store_master
store_master = pd.DataFrame({
    'store_id':          ['S01', 'S02', 'S03'],
    'store_taxonomies':  ['community', 'mall', 'tourist'],
    'province':          ['Chiang Mai', 'Bangkok', 'Phuket'],
    'store_size_sqm':    [250, 600, 180],
    'opened_year':       [2018, 2015, 2021]
})
print(f"✅ store_master:   {store_master.shape}")

# 3. warehouse_master
warehouse_master = pd.DataFrame({
    'warehouse_id':           ['W01', 'W02'],
    'warehouse_taxonomies':   ['central_dc', 'north_dc'],
    'province':               ['Bangkok', 'Chiang Mai'],
    'capacity_units':         [50000, 20000]
})
print(f"✅ warehouse_master: {warehouse_master.shape}")

# 4. customer_master
customer_types = ['regular', 'new', 'price_sensitive', 'premium', 'occasional']
customer_master = pd.DataFrame({
    'customer_id':         [f'C{str(i+1).zfill(5)}' for i in range(N_CUSTOMERS)],
    'customer_taxonomies': np.random.choice(customer_types, N_CUSTOMERS, p=[0.35,0.2,0.25,0.1,0.1])
})
print(f"✅ customer_master: {customer_master.shape}")


In [ ]:

# ── Transactional tables ──────────────────────────────────────────────────────

# 5. promotion_master
promo_data = []
promo_periods = [
    ('PROMO001', '2025-02-10', '2025-02-14', 'Valentine'),
    ('PROMO002', '2025-04-13', '2025-04-15', 'Songkran'),
    ('PROMO003', '2025-06-01', '2025-06-07', 'MidYear'),
    ('PROMO004', '2025-10-31', '2025-11-03', 'Halloween'),
    ('PROMO005', '2025-11-11', '2025-11-11', '11-11'),
    ('PROMO006', '2025-11-26', '2025-12-03', 'BlackFriday'),
    ('PROMO007', '2025-12-20', '2025-12-31', 'Christmas')
]
for pid, sd, ed, name in promo_periods:
    n_products = np.random.randint(5, 15)
    prods = np.random.choice(product_master['product_id'], n_products, replace=False)
    for prod in prods:
        promo_data.append({
            'promotion_id':   pid,
            'promotion_name': name,
            'product_id':     prod,
            'discount':       round(np.random.choice([0.05, 0.10, 0.15, 0.20]), 2),
            'start_date':     sd,
            'end_date':       ed
        })
promotion_master = pd.DataFrame(promo_data)
print(f"✅ promotion_master: {promotion_master.shape}")

# 6. purchasing_order (PO)
po_records = []
dates = pd.date_range(START_DATE, END_DATE, freq='W')
po_id = 1
for dt in dates:
    n_po = np.random.randint(3, 8)
    selected_products = np.random.choice(product_master['product_id'], n_po, replace=False)
    wh = np.random.choice(['W01','W02'])
    for prod in selected_products:
        row = product_master[product_master['product_id']==prod].iloc[0]
        qty = np.random.randint(50, 400)
        cost_per = round(row['price'] * (1 - row['gross_margin_pct']) * np.random.uniform(0.95,1.05), 2)
        shelf = int(row['shelf_life_days'])
        arrival_date = dt + timedelta(days=np.random.randint(1,3))
        # Keep mock PO dates realistic: even short shelf-life items should arrive before expiry.
        max_mfg_lag = max(0, min(14, shelf - 3))
        mfg_date = dt - timedelta(days=np.random.randint(0, max_mfg_lag + 1))
        expire_date = mfg_date + timedelta(days=shelf)
        if expire_date <= arrival_date:
            expire_date = arrival_date + timedelta(days=max(1, min(7, shelf // 2)))
        po_records.append({
            'po_id':               f'PO{str(po_id).zfill(6)}',
            'warehouse_id':        wh,
            'product_id':          prod,
            'qty':                 qty,
            'po_price_per_unit':   cost_per,
            'po_price_total':      round(cost_per * qty, 2),
            'unit':                'pcs',
            'manufacturing_date':  mfg_date.date(),
            'po_date':             dt.date(),
            'arrival_date':        arrival_date.date(),
            'expire_date':         expire_date.date()
        })
        po_id += 1

purchasing_order = pd.DataFrame(po_records)
print(f"✅ purchasing_order: {purchasing_order.shape}")


In [ ]:

# 7. sales_transaction (main fact table — realistic demand patterns)
sales_records = []

store_demand_scale = {'S01': 1.0, 'S02': 1.8, 'S03': 0.7}
cat_base_demand    = {'Beverage':4, 'Snack':3, 'Fresh Food':5, 'Personal Care':2, 'Household':1}

promo_df = promotion_master.copy()
promo_df['start_date'] = pd.to_datetime(promo_df['start_date'])
promo_df['end_date']   = pd.to_datetime(promo_df['end_date'])

all_dates = pd.date_range(START_DATE, END_DATE, freq='D')

for dt in all_dates:
    dow = dt.dayofweek   # 0=Mon
    month = dt.month
    is_weekend = dow >= 5

    weekend_mult = 1.4 if is_weekend else 1.0
    season_mult  = 1.0 + 0.15*np.sin((month - 3) * np.pi / 6)

    for store in ['S01','S02','S03']:
        n_transactions = int(np.random.poisson(
            (25 if store=='S02' else 12 if store=='S01' else 8) * weekend_mult * season_mult
        ))
        for _ in range(n_transactions):
            prod = np.random.choice(product_master['product_id'])
            row  = product_master[product_master['product_id']==prod].iloc[0]
            base = cat_base_demand[row['product_taxonomies']]
            qty  = max(1, int(np.random.poisson(base * store_demand_scale[store])))

            # Check active promo
            active = promo_df[
                (promo_df['product_id']==prod) &
                (promo_df['start_date']<=dt) &
                (promo_df['end_date']>=dt)
            ]
            disc = 0.0
            promo_id = None
            if len(active) > 0:
                disc     = float(active.iloc[0]['discount'])
                promo_id = active.iloc[0]['promotion_id']
                qty      = int(qty * (1 + disc * 2.5))  # promo uplift

            price = round(row['price'] * (1 - disc), 2)
            cust  = np.random.choice(customer_master['customer_id'])

            # Link to a PO
            po_options = purchasing_order[purchasing_order['product_id']==prod]['po_id']
            po_ref = np.random.choice(po_options) if len(po_options)>0 else None

            ts = dt + timedelta(
                hours=int(np.random.uniform(9,21)),
                minutes=int(np.random.randint(0,60))
            )
            sales_records.append({
                'datetime':     ts,
                'product_id':   prod,
                'store_id':     store,
                'customer_id':  cust,
                'qty':          qty,
                'price':        price,
                'promotion_id': promo_id,
                'po_id':        po_ref
            })

sales_transaction = pd.DataFrame(sales_records)
sales_transaction['datetime'] = pd.to_datetime(sales_transaction['datetime'])
print(f"✅ sales_transaction: {sales_transaction.shape}")
print(f"   Total revenue: {(sales_transaction['qty']*sales_transaction['price']).sum():,.2f} THB")


In [ ]:

# 8. stock_movement_transaction
stock_records = []
for _, po in purchasing_order.iterrows():
    stores = np.random.choice(['S01','S02','S03'], size=np.random.randint(1,4), replace=False)
    remaining = po['qty']
    for s in stores:
        if remaining <= 0: break
        transfer_qty = np.random.randint(10, max(11, remaining//len(stores)+20))
        transfer_qty = min(transfer_qty, remaining)
        remaining -= transfer_qty
        arrive = pd.to_datetime(po['arrival_date'])
        stock_records.append({
            'po_id':         po['po_id'],
            'warehouse_id':  po['warehouse_id'],
            'store_id':      s,
            'qty':           transfer_qty,
            'receive_date':  (arrive + timedelta(days=1)).date(),
            'transfer_date': arrive.date()
        })

stock_movement = pd.DataFrame(stock_records)
print(f"✅ stock_movement_transaction: {stock_movement.shape}")

# ── Save all tables ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
tables = {
    'sales_transaction':       sales_transaction,
    'purchasing_order':        purchasing_order,
    'product_master':          product_master,
    'promotion_master':        promotion_master,
    'store_master':            store_master,
    'customer_master':         customer_master,
    'warehouse_master':        warehouse_master,
    'stock_movement_transaction': stock_movement
}
for name, df in tables.items():
    df.to_csv(f'data/{name}.csv', index=False)

print("\n📁 All 8 tables saved to data/")
for name, df in tables.items():
    print(f"   {name:40s}: {df.shape[0]:>6,} rows × {df.shape[1]} cols")


---
## 2. ✅ Data Validation & Quality Checks

In [ ]:

def validate_table(df, name, checks=None):
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Shape       : {df.shape[0]:,} rows × {df.shape[1]} cols")
    null_pct = df.isnull().mean() * 100
    if null_pct.max() > 0:
        print(f"  Null cols   : {null_pct[null_pct>0].to_dict()}")
    else:
        print(f"  Null cols   : None ✓")
    dups = df.duplicated().sum()
    print(f"  Duplicates  : {dups} {'✓' if dups==0 else '⚠️'}")
    if checks:
        for label, result in checks.items():
            print(f"  {label:20s}: {result}")

validate_table(sales_transaction, "sales_transaction", {
    "Negative qty"   : f"{(sales_transaction['qty']<0).sum()} rows",
    "Negative price" : f"{(sales_transaction['price']<=0).sum()} rows",
    "Date range"     : f"{sales_transaction['datetime'].min().date()} → {sales_transaction['datetime'].max().date()}"
})

validate_table(purchasing_order, "purchasing_order", {
    "Expire < Arrive": f"{(pd.to_datetime(purchasing_order['expire_date']) < pd.to_datetime(purchasing_order['arrival_date'])).sum()} rows",
    "Negative qty"   : f"{(purchasing_order['qty']<0).sum()} rows",
    "Negative cost"  : f"{(purchasing_order['po_price_per_unit']<=0).sum()} rows"
})

validate_table(product_master,   "product_master")
validate_table(promotion_master, "promotion_master", {
    "Date logic OK": f"{(pd.to_datetime(promotion_master['end_date']) >= pd.to_datetime(promotion_master['start_date'])).all()}"
})
validate_table(stock_movement, "stock_movement_transaction", {
    "Negative qty": f"{(stock_movement['qty']<0).sum()} rows"
})

print("\n✅ Data Validation Complete")


---
## 3. 📊 Exploratory Data Analysis (EDA)

In [ ]:

# Prepare aggregated views
sales_transaction['date']    = pd.to_datetime(sales_transaction['datetime']).dt.date
sales_transaction['date']    = pd.to_datetime(sales_transaction['date'])
sales_transaction['revenue'] = sales_transaction['qty'] * sales_transaction['price']
sales_transaction['month']   = sales_transaction['date'].dt.to_period('M')
sales_transaction['weekday'] = sales_transaction['date'].dt.day_name()
sales_transaction = sales_transaction.merge(product_master[['product_id','product_taxonomies','gross_margin_pct']], on='product_id', how='left')

# Daily revenue by store
daily_rev = sales_transaction.groupby(['date','store_id'])['revenue'].sum().reset_index()
monthly_rev = sales_transaction.groupby(['month','store_id'])['revenue'].sum().reset_index()
monthly_rev['month_str'] = monthly_rev['month'].astype(str)

print("Revenue summary by store:")
store_rev = sales_transaction.groupby('store_id').agg(
    total_revenue=('revenue','sum'),
    total_qty=('qty','sum'),
    n_transactions=('qty','count')
).round(2)
store_rev = store_rev.merge(store_master[['store_id','store_taxonomies','province']], on='store_id')
print(store_rev.to_string())


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SME Retail — Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

# Plot 1: Monthly revenue by store
ax = axes[0,0]
for sid, grp in monthly_rev.groupby('store_id'):
    ax.plot(grp['month_str'], grp['revenue']/1000, marker='o', label=sid, linewidth=2)
ax.set_title('Monthly Revenue by Store (THB K)', fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (K THB)')
ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(alpha=0.3)

# Plot 2: Revenue by category
ax = axes[0,1]
cat_rev = sales_transaction.groupby('product_taxonomies')['revenue'].sum().sort_values(ascending=True)
bars = ax.barh(cat_rev.index, cat_rev.values/1000, color=COLORS['pastel'][:len(cat_rev)])
ax.set_title('Revenue by Product Category (THB K)', fontweight='bold')
ax.set_xlabel('Revenue (K THB)'); ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, cat_rev.values):
    ax.text(val/1000+5, bar.get_y()+bar.get_height()/2, f'{val/1000:.0f}K', va='center', fontsize=9)

# Plot 3: Weekday sales pattern
ax = axes[0,2]
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
wd_rev = sales_transaction.groupby('weekday')['revenue'].mean().reindex(order)
ax.bar(range(7), wd_rev.values/1000, color=[COLORS['primary'] if i<5 else COLORS['accent'] for i in range(7)])
ax.set_xticks(range(7)); ax.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=30)
ax.set_title('Avg Daily Revenue by Weekday (THB K)', fontweight='bold')
ax.set_ylabel('Avg Revenue (K THB)'); ax.grid(axis='y', alpha=0.3)

# Plot 4: Store revenue share pie
ax = axes[1,0]
sr = sales_transaction.groupby('store_id')['revenue'].sum()
ax.pie(sr.values, labels=[f"{i}\n{store_master[store_master.store_id==i]['province'].values[0]}" for i in sr.index],
       autopct='%1.1f%%', colors=COLORS['pastel'][:3], startangle=140,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Revenue Share by Store', fontweight='bold')

# Plot 5: Promotion vs Non-promo revenue
ax = axes[1,1]
promo_flag = sales_transaction['promotion_id'].notna()
promo_comp = sales_transaction.groupby(promo_flag)['revenue'].sum()
labels = ['Non-Promo', 'Promo']
colors = [COLORS['primary'], COLORS['accent']]
ax.bar(labels, promo_comp.values/1000, color=colors, width=0.5, edgecolor='white', linewidth=2)
ax.set_title('Revenue: Promo vs Non-Promo (THB K)', fontweight='bold')
ax.set_ylabel('Revenue (K THB)'); ax.grid(axis='y', alpha=0.3)
for i, (lbl, val) in enumerate(zip(labels, promo_comp.values)):
    ax.text(i, val/1000+10, f'{val/1000:.0f}K', ha='center', fontweight='bold')

# Plot 6: Product margin distribution
ax = axes[1,2]
ax.hist(product_master['gross_margin_pct'], bins=15, color=COLORS['secondary'], alpha=0.8, edgecolor='white')
ax.axvline(product_master['gross_margin_pct'].mean(), color=COLORS['accent'], linestyle='--', linewidth=2,
           label=f"Mean={product_master['gross_margin_pct'].mean():.2%}")
ax.set_title('Gross Margin % Distribution', fontweight='bold')
ax.set_xlabel('Gross Margin %'); ax.set_ylabel('Count'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ EDA charts saved")


In [ ]:

# Customer segmentation
cust_rev = sales_transaction.groupby('customer_id').agg(
    total_spend=('revenue','sum'),
    n_visits=('date', lambda x: x.nunique()),
    avg_basket=('revenue','mean')
).reset_index()
cust_rev = cust_rev.merge(customer_master, on='customer_id')

print("Customer Segment Summary:")
seg = cust_rev.groupby('customer_taxonomies').agg(
    n_customers=('customer_id','count'),
    avg_spend=('total_spend','mean'),
    avg_visits=('n_visits','mean')
).round(2)
print(seg.to_string())

# Expiry risk analysis
purchasing_order['expire_date_dt']  = pd.to_datetime(purchasing_order['expire_date'])
purchasing_order['arrival_date_dt'] = pd.to_datetime(purchasing_order['arrival_date'])
ref_date = pd.Timestamp('2025-12-31')
purchasing_order['days_to_expire']  = (purchasing_order['expire_date_dt'] - ref_date).dt.days

expiry_risk = purchasing_order[purchasing_order['days_to_expire'] < 30].copy()
expiry_risk = expiry_risk.merge(product_master[['product_id','product_taxonomies','price']], on='product_id')
expiry_risk['at_risk_value'] = expiry_risk['qty'] * expiry_risk['po_price_per_unit']
print(f"\n⚠️  Expiry Risk (expire within 30 days of 2025-12-31):")
print(f"   {len(expiry_risk)} POs at risk, total value: {expiry_risk['at_risk_value'].sum():,.2f} THB")


---
## 4. 🔧 Feature Engineering

In [ ]:

# Aggregate to daily store-product level (the modeling grain)
daily = sales_transaction.groupby(['date','store_id','product_id']).agg(
    daily_qty=('qty','sum'),
    daily_revenue=('revenue','sum')
).reset_index()

daily = daily.merge(product_master[['product_id','product_taxonomies','price','shelf_life_days','gross_margin_pct']], on='product_id', how='left')
daily = daily.merge(store_master[['store_id','store_taxonomies','province']], on='store_id', how='left')

# Calendar features
daily['year']      = daily['date'].dt.year
daily['month']     = daily['date'].dt.month
daily['weekday']   = daily['date'].dt.dayofweek
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['is_weekend']  = (daily['weekday'] >= 5).astype(int)
daily['is_month_end'] = daily['date'].dt.is_month_end.astype(int)
daily['quarter']   = daily['date'].dt.quarter

# Promotion feature
promo_df2 = promotion_master.copy()
promo_df2['start_date'] = pd.to_datetime(promo_df2['start_date'])
promo_df2['end_date']   = pd.to_datetime(promo_df2['end_date'])

def get_discount(row):
    matches = promo_df2[
        (promo_df2['product_id'] == row['product_id']) &
        (promo_df2['start_date'] <= row['date']) &
        (promo_df2['end_date']   >= row['date'])
    ]
    return matches['discount'].max() if len(matches) > 0 else 0.0

daily['discount_pct']   = daily.apply(get_discount, axis=1)
daily['is_promo']       = (daily['discount_pct'] > 0).astype(int)

# Lag & rolling features (per store-product group)
daily = daily.sort_values(['store_id','product_id','date'])
for lag in [1, 7, 14]:
    daily[f'lag_{lag}d_qty'] = daily.groupby(['store_id','product_id'])['daily_qty'].shift(lag)
for window in [7, 14, 30]:
    daily[f'roll_{window}d_qty'] = (
        daily.groupby(['store_id','product_id'])['daily_qty']
             .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# Categorical encoding
le_cat   = LabelEncoder()
le_store = LabelEncoder()
le_prod  = LabelEncoder()
daily['cat_enc']   = le_cat.fit_transform(daily['product_taxonomies'].fillna('Unknown'))
daily['store_enc'] = le_store.fit_transform(daily['store_id'])
daily['prod_enc']  = le_prod.fit_transform(daily['product_id'])

daily_clean = daily.dropna(subset=['lag_1d_qty','roll_7d_qty']).copy()
print(f"✅ Feature matrix: {daily_clean.shape}")
print(f"   Features: {[c for c in daily_clean.columns if c not in ['date','store_id','product_id','daily_qty','daily_revenue']]}")


---
## 5. 🤖 Demand Forecasting: Baseline vs ML Model

In [ ]:

# Feature set for modeling
FEATURE_COLS = [
    'month','weekday','day_of_year','is_weekend','is_month_end','quarter',
    'discount_pct','is_promo','lag_1d_qty','lag_7d_qty','lag_14d_qty',
    'roll_7d_qty','roll_14d_qty','roll_30d_qty',
    'price','gross_margin_pct','shelf_life_days',
    'cat_enc','store_enc','prod_enc'
]
TARGET = 'daily_qty'

# Time-based train/test split: train on first 10 months, test on last 2
cutoff = pd.Timestamp('2025-11-01')
train  = daily_clean[daily_clean['date'] < cutoff]
test   = daily_clean[daily_clean['date'] >= cutoff]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train[TARGET]
X_test  = test[FEATURE_COLS].fillna(0)
y_test  = test[TARGET]

print(f"Train size: {X_train.shape}  |  Test size: {X_test.shape}")
print(f"Train period: {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Test  period: {test['date'].min().date()}  → {test['date'].max().date()}")

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

# ── Baseline: 14-day rolling average ──────────────────────────────────────────
baseline_pred = test['roll_14d_qty'].fillna(test['roll_7d_qty']).fillna(y_train.mean())
baseline_wape = wape(y_test.values, baseline_pred.values)
baseline_mae  = mean_absolute_error(y_test, baseline_pred)
print(f"\n📌 Baseline (14-day rolling avg):")
print(f"   WAPE = {baseline_wape:.4f}  |  MAE = {baseline_mae:.4f}")


In [ ]:
# ── ML Model Candidates + Honest Model Selection ────────────────────────────
model_candidates = {
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, min_samples_leaf=10, random_state=42
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=300, max_depth=14, min_samples_leaf=5,
        n_jobs=-1, random_state=42
    )
}

model_results = []
trained_models = {}
for name, model in model_candidates.items():
    model.fit(X_train, y_train)
    pred = np.maximum(0, model.predict(X_test))
    model_results.append({
        'model': name,
        'WAPE': wape(y_test.values, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'prediction': pred
    })
    trained_models[name] = model

model_results_df = pd.DataFrame(model_results).sort_values('WAPE').reset_index(drop=True)
print("\n🤖 ML Candidate Results:")
print(model_results_df[['model','WAPE','MAE','RMSE']].to_string(index=False, formatters={
    'WAPE': '{:.4f}'.format, 'MAE': '{:.4f}'.format, 'RMSE': '{:.4f}'.format
}))

best_row = model_results_df.iloc[0]
selected_model_name = best_row['model']
selected_model = trained_models[selected_model_name]
selected_pred = best_row['prediction']
selected_wape = float(best_row['WAPE'])
selected_mae = float(best_row['MAE'])
selected_rmse = float(best_row['RMSE'])
improvement = baseline_wape - selected_wape
improvement_pct = improvement / baseline_wape * 100
model_status = 'APPROVED' if improvement > 0 else 'CANDIDATE_NEEDS_IMPROVEMENT'

print(f"\n🏆 Selected ML model: {selected_model_name}")
print(f"   WAPE = {selected_wape:.4f}  |  MAE = {selected_mae:.4f}  |  RMSE = {selected_rmse:.4f}")
if improvement > 0:
    print(f"📈 ML beats baseline by {improvement:.4f} WAPE ({improvement_pct:.1f}% better)")
else:
    print(f"⚠️ Baseline is still stronger by {-improvement:.4f} WAPE ({-improvement_pct:.1f}%). Keep ML as a candidate, not production.")

# Backward-compatible aliases for later notebook sections.
gb_model = selected_model
gb_pred = selected_pred
gb_wape = selected_wape
gb_mae = selected_mae

# ── Feature Importance ────────────────────────────────────────────────────────
feat_imp = pd.Series(selected_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(f"\nTop 10 features ({selected_model_name}):")
print(feat_imp.head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Demand Forecasting Results', fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted (sample store-product)
ax = axes[0]
sample = test[(test['store_id']=='S02') & (test['product_id']=='P0001')].copy()
sample = sample.sort_values('date')
if len(sample) > 5:
    idx = sample.index
    sample_pred = selected_pred[test.index.get_indexer(idx)]
    ax.plot(sample['date'], sample['daily_qty'], label='Actual', marker='o', linewidth=2)
    ax.plot(sample['date'], sample_pred, label=f'{selected_model_name} Forecast', color=COLORS['accent'], linewidth=2, linestyle='--')
    ax.set_title('Sample Forecast: S02-P0001')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
else:
    ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Sample Forecast')

# Plot 2: Model comparison
ax = axes[1]
models_comp = ['Baseline (14d Roll)'] + model_results_df['model'].tolist()
wapes = [baseline_wape] + model_results_df['WAPE'].tolist()
bar_colors = [COLORS['secondary']] + [COLORS['primary'] if m == selected_model_name else COLORS['pastel'][2] for m in model_results_df['model']]
ax.bar(models_comp, wapes, color=bar_colors, alpha=0.85)
ax.set_ylabel('WAPE (lower is better)')
ax.set_title('Forecast Accuracy Comparison')
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(wapes):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

# Plot 3: Feature importance
ax = axes[2]
top10 = feat_imp.head(10)
ax.barh(range(len(top10)), top10.values, color=COLORS['primary'], alpha=0.8)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index)
ax.invert_yaxis()
ax.set_title(f'Top 10 Feature Importance ({selected_model_name})')

plt.tight_layout()
plt.savefig('forecast_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. 📦 Inventory Optimization & Reorder Recommendation

In [ ]:

# Current stock on hand (approximate from stock_movement - sales)
stock_in = stock_movement.groupby(['store_id','po_id'])['qty'].sum().reset_index()
stock_in = stock_in.merge(purchasing_order[['po_id','product_id']], on='po_id', how='left')
stock_in_agg = stock_in.groupby(['store_id','product_id'])['qty'].sum().reset_index()
stock_in_agg.rename(columns={'qty':'total_received'}, inplace=True)

sales_out = sales_transaction.groupby(['store_id','product_id'])['qty'].sum().reset_index()
sales_out.rename(columns={'qty':'total_sold'}, inplace=True)

stock_status = stock_in_agg.merge(sales_out, on=['store_id','product_id'], how='outer').fillna(0)
stock_status['current_stock'] = (stock_status['total_received'] - stock_status['total_sold']).clip(lower=0)

# 7-day forecast per store-product (using selected ML model on test period tail)
last_known = daily_clean[daily_clean['date'] >= pd.Timestamp('2025-11-01')]
forecast_7d = last_known.groupby(['store_id','product_id']).apply(
    lambda g: np.maximum(0, selected_model.predict(g[FEATURE_COLS].fillna(0))).mean() * 7
).reset_index()
forecast_7d.columns = ['store_id','product_id','forecast_7d_qty']

# Reorder logic
SAFETY_STOCK_DAYS = 3
LEAD_TIME_DAYS    = 2

reorder = stock_status.merge(forecast_7d, on=['store_id','product_id'], how='left')
reorder['forecast_7d_qty'] = reorder['forecast_7d_qty'].fillna(
    stock_status['total_sold'] / 365 * 7  # fallback: avg daily × 7
)
reorder = reorder.merge(product_master[['product_id','price','product_taxonomies','gross_margin_pct','shelf_life_days']], on='product_id', how='left')
reorder = reorder.merge(store_master[['store_id','store_taxonomies','province']], on='store_id', how='left')

daily_demand_est = reorder['forecast_7d_qty'] / 7
reorder['safety_stock']    = daily_demand_est * SAFETY_STOCK_DAYS
reorder['reorder_point']   = daily_demand_est * LEAD_TIME_DAYS + reorder['safety_stock']
reorder['reorder_qty']     = (reorder['forecast_7d_qty'] + reorder['safety_stock'] - reorder['current_stock']).clip(lower=0).round().astype(int)
reorder['revenue_opportunity'] = (reorder['reorder_qty'] * reorder['price']).round(2)
reorder['stockout_risk']   = (reorder['current_stock'] < reorder['reorder_point']).map({True:'HIGH', False:'LOW'})

# Filter actionable
recommendations = reorder[reorder['reorder_qty'] > 0].sort_values('revenue_opportunity', ascending=False)
recommendations = recommendations[['store_id','province','product_id','product_taxonomies',
                                    'current_stock','forecast_7d_qty','reorder_qty',
                                    'revenue_opportunity','stockout_risk','gross_margin_pct']].reset_index(drop=True)

print(f"📋 Reorder Recommendations: {len(recommendations)} store-product pairs")
print(f"   Total revenue opportunity: {recommendations['revenue_opportunity'].sum():,.2f} THB")
print(f"   HIGH stockout risk items:  {(recommendations['stockout_risk']=='HIGH').sum()}")
print("\nTop 15 Priority Reorders:")
print(recommendations.head(15).to_string(index=False))


In [ ]:

# Save recommendation file
recommendations.to_csv('data/model_recommendation_output.csv', index=False)
print("✅ Saved: data/model_recommendation_output.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Inventory Optimization Dashboard', fontsize=14, fontweight='bold')

# Plot 1: Revenue opportunity by store
ax = axes[0]
rev_by_store = recommendations.groupby('store_id')['revenue_opportunity'].sum()
ax.bar(rev_by_store.index, rev_by_store.values/1000, color=COLORS['pastel'][:3], edgecolor='white', linewidth=2)
ax.set_title('Revenue Opportunity by Store (THB K)', fontweight='bold')
ax.set_ylabel('THB K'); ax.grid(axis='y', alpha=0.3)
for i, (s, v) in enumerate(rev_by_store.items()):
    ax.text(i, v/1000+0.5, f'{v/1000:.1f}K', ha='center', fontweight='bold')

# Plot 2: Top 10 products by revenue opportunity
ax = axes[1]
top_prods = recommendations.groupby('product_id')['revenue_opportunity'].sum().nlargest(10).sort_values()
ax.barh(top_prods.index, top_prods.values/1000, color=COLORS['primary'], alpha=0.85)
ax.set_title('Top 10 Products: Revenue Opportunity', fontweight='bold')
ax.set_xlabel('THB K'); ax.grid(axis='x', alpha=0.3)

# Plot 3: Stockout risk by category
ax = axes[2]
risk_cat = recommendations[recommendations['stockout_risk']=='HIGH'].groupby('product_taxonomies')['product_id'].count()
if len(risk_cat) > 0:
    ax.pie(risk_cat.values, labels=risk_cat.index, autopct='%1.0f%%',
           colors=COLORS['pastel'][:len(risk_cat)], startangle=140,
           wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('HIGH Stockout Risk by Category', fontweight='bold')

plt.tight_layout()
plt.savefig('inventory_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 7. 🎯 Promotion Uplift Analysis

In [ ]:

# Compare sales during promotion vs equivalent non-promo period (same weekday/store/product)
promo_df3 = promotion_master.copy()
promo_df3['start_date'] = pd.to_datetime(promo_df3['start_date'])
promo_df3['end_date']   = pd.to_datetime(promo_df3['end_date'])

uplift_records = []
for _, promo in promo_df3.groupby('promotion_id').agg({'start_date':'first','end_date':'first'}).reset_index().iterrows():
    pid = promo['promotion_id']
    sd  = promo['start_date']
    ed  = promo['end_date']
    duration = (ed - sd).days + 1

    promo_products = promo_df3[promo_df3['promotion_id']==pid]['product_id'].tolist()
    promo_disc     = promo_df3[promo_df3['promotion_id']==pid].set_index('product_id')['discount'].to_dict()

    pre_start = sd - timedelta(days=duration*2)
    pre_end   = sd - timedelta(days=1)

    for prod in promo_products:
        disc = promo_disc.get(prod, 0)
        cost_price = product_master[product_master['product_id']==prod]['price'].values
        if len(cost_price)==0: continue
        full_price = cost_price[0]

        promo_sales = daily_clean[
            (daily_clean['product_id']==prod) &
            (daily_clean['date']>=sd) & (daily_clean['date']<=ed)
        ]['daily_qty'].mean()

        pre_sales = daily_clean[
            (daily_clean['product_id']==prod) &
            (daily_clean['date']>=pre_start) & (daily_clean['date']<=pre_end)
        ]['daily_qty'].mean()

        if pd.isna(pre_sales) or pre_sales == 0: continue

        uplift_pct    = (promo_sales - pre_sales) / pre_sales
        incremental   = max(0, promo_sales - pre_sales) * duration
        revenue_gain  = incremental * full_price * (1 - disc)
        margin_gain   = revenue_gain * (product_master[product_master['product_id']==prod]['gross_margin_pct'].values[0] - disc)
        discount_cost = promo_sales * full_price * disc * duration

        uplift_records.append({
            'promotion_id':   pid,
            'product_id':     prod,
            'discount_pct':   disc,
            'pre_avg_daily':  round(pre_sales, 2),
            'promo_avg_daily': round(promo_sales, 2),
            'uplift_pct':     round(uplift_pct * 100, 1),
            'incremental_units': round(incremental, 1),
            'revenue_gain_thb': round(revenue_gain, 2),
            'margin_gain_thb':  round(margin_gain, 2),
            'discount_cost_thb': round(discount_cost, 2),
            'roi':             round(margin_gain / max(1, discount_cost), 3)
        })

uplift_df = pd.DataFrame(uplift_records)
print(f"✅ Promotion uplift analysis: {len(uplift_df)} promo-product pairs")
print(f"\nTop 10 highest-ROI promotions:")
print(uplift_df.sort_values('roi', ascending=False).head(10)[
    ['promotion_id','product_id','discount_pct','uplift_pct','incremental_units','margin_gain_thb','roi']
].to_string(index=False))

# Promo summary
promo_summary = uplift_df.groupby('promotion_id').agg(
    avg_uplift=('uplift_pct','mean'),
    total_incremental=('incremental_units','sum'),
    total_margin_gain=('margin_gain_thb','sum'),
    total_discount_cost=('discount_cost_thb','sum'),
    avg_roi=('roi','mean')
).round(2)
print(f"\nPromotion Summary:")
print(promo_summary.to_string())


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Promotion Uplift Analysis', fontsize=14, fontweight='bold')

# Plot 1: Avg uplift by promotion
ax = axes[0]
pu = promo_summary['avg_uplift'].sort_values(ascending=True)
colors_bar = [COLORS['success'] if v > 0 else COLORS['neutral'] for v in pu.values]
ax.barh(pu.index, pu.values, color=colors_bar, alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Avg Sales Uplift % by Promotion', fontweight='bold')
ax.set_xlabel('Uplift %'); ax.grid(axis='x', alpha=0.3)

# Plot 2: ROI scatter
ax = axes[1]
sc = ax.scatter(uplift_df['discount_pct']*100, uplift_df['roi'],
                c=uplift_df['margin_gain_thb'], cmap='RdYlGn',
                s=60, alpha=0.7, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Margin Gain (THB)')
ax.axhline(1, color='red', linestyle='--', linewidth=1.5, label='ROI=1 breakeven')
ax.set_title('Discount % vs ROI (color=margin gain)', fontweight='bold')
ax.set_xlabel('Discount %'); ax.set_ylabel('ROI'); ax.legend(); ax.grid(alpha=0.3)

# Plot 3: Margin gain vs discount cost
ax = axes[2]
ax.scatter(promo_summary['total_discount_cost'], promo_summary['total_margin_gain'],
           s=200, color=COLORS['secondary'], alpha=0.8, zorder=5)
for pid in promo_summary.index:
    ax.annotate(pid, (promo_summary.loc[pid,'total_discount_cost'],
                      promo_summary.loc[pid,'total_margin_gain']),
                textcoords='offset points', xytext=(5,3), fontsize=8)
ax.plot([0, promo_summary['total_discount_cost'].max()],
        [0, promo_summary['total_discount_cost'].max()], 'r--', label='Breakeven')
ax.set_title('Margin Gain vs Discount Cost', fontweight='bold')
ax.set_xlabel('Discount Cost (THB)'); ax.set_ylabel('Margin Gain (THB)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('promotion_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 8. ⚙️ MLOps Monitoring: Drift Detection & Model Registry

In [ ]:

# ── Simulate weekly WAPE tracking (model monitoring) ─────────────────────────
weeks = pd.date_range('2025-11-01', '2025-12-31', freq='W')
wape_history = []
for i, wk in enumerate(weeks):
    wk_start = wk
    wk_end   = wk + timedelta(days=6)
    wk_data  = daily_clean[(daily_clean['date']>=wk_start) & (daily_clean['date']<=wk_end)]
    if len(wk_data) < 10: continue
    X_wk = wk_data[FEATURE_COLS].fillna(0)
    y_wk = wk_data[TARGET]
    pred_wk = np.maximum(0, selected_model.predict(X_wk))
    base_wk = wk_data['roll_14d_qty'].fillna(0)
    w = wape(y_wk.values, pred_wk)
    b = wape(y_wk.values, base_wk.values)
    drift = abs(wk_data['daily_qty'].mean() - train['daily_qty'].mean()) / train['daily_qty'].mean()
    wape_history.append({
        'week': wk_start.date(),
        'model_wape': round(w, 4),
        'baseline_wape': round(b, 4),
        'data_drift_pct': round(drift*100, 2),
        'retraining_triggered': 'YES' if w > 0.50 else 'NO'
    })

wape_monitor = pd.DataFrame(wape_history)
print("MLOps Weekly Monitoring Log:")
print(wape_monitor.to_string(index=False))

# ── Model registry metadata ───────────────────────────────────────────────────
registry = {
    'model_name':     f'SME_Retail_{selected_model_name}_v1',
    'version':        '1.0.0',
    'train_date':     str(datetime.now().date()),
    'train_period':   '2025-01-01 → 2025-10-31',
    'test_period':    '2025-11-01 → 2025-12-31',
    'train_rows':     int(len(train)),
    'test_rows':      int(len(test)),
    'features':       FEATURE_COLS,
    'model_type':     selected_model.__class__.__name__,
    'hyperparams':    selected_model.get_params(),
    'metrics': {
        'baseline_wape': float(round(baseline_wape, 4)),
        'model_wape':    float(round(selected_wape, 4)),
        'model_mae':     float(round(selected_mae, 4)),
        'model_rmse':    float(round(selected_rmse, 4)),
        'wape_improvement': float(round(improvement, 4)),
        'wape_improvement_pct': float(round(improvement_pct, 2))
    },
    'retraining_policy': 'Weekly if WAPE > 0.50 or drift > 20%',
    'status':         model_status
}

os.makedirs('mlops', exist_ok=True)
os.makedirs('models', exist_ok=True)
joblib.dump(selected_model, 'models/demand_forecast_model.joblib')
with open('models/feature_schema.json', 'w') as f:
    json.dump({'feature_columns': FEATURE_COLS, 'target': TARGET}, f, indent=2)
with open('mlops/model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2, default=str)

print("\n✅ Model Registry & artifacts saved:")
for k, v in registry.items():
    if k not in ['features', 'hyperparams']:
        print(f"   {k:25s}: {v}")
print("   model_artifact          : models/demand_forecast_model.joblib")
print("   feature_schema          : models/feature_schema.json")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MLOps Monitoring Dashboard', fontsize=14, fontweight='bold')

# WAPE over time
ax = axes[0]
ax.plot(range(len(wape_monitor)), wape_monitor['model_wape'], 'o-',
        color=COLORS['primary'], linewidth=2, label='GBM WAPE', markersize=7)
ax.plot(range(len(wape_monitor)), wape_monitor['baseline_wape'], 's--',
        color='gray', linewidth=1.5, label='Baseline WAPE', markersize=6)
ax.axhline(0.50, color='red', linestyle=':', linewidth=1.5, label='Retraining Threshold')
ax.set_xticks(range(len(wape_monitor)))
ax.set_xticklabels([str(w) for w in wape_monitor['week']], rotation=40, ha='right', fontsize=8)
ax.set_title('Weekly WAPE Monitoring', fontweight='bold')
ax.set_ylabel('WAPE'); ax.legend(); ax.grid(alpha=0.3)

# Data drift
ax = axes[1]
colors_drift = [COLORS['success'] if v < 20 else COLORS['accent'] for v in wape_monitor['data_drift_pct']]
ax.bar(range(len(wape_monitor)), wape_monitor['data_drift_pct'], color=colors_drift, alpha=0.8)
ax.axhline(20, color='red', linestyle='--', linewidth=1.5, label='Drift Threshold 20%')
ax.set_xticks(range(len(wape_monitor)))
ax.set_xticklabels([str(w) for w in wape_monitor['week']], rotation=40, ha='right', fontsize=8)
ax.set_title('Data Drift % (Weekly)', fontweight='bold')
ax.set_ylabel('Drift %'); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('mlops_monitoring.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 9. 📋 Final Output: Action List + Executive Summary

In [ ]:

# ── Action List (Top 20 reorder recommendations) ─────────────────────────────
action_list = recommendations.head(20).copy()
action_list['action'] = action_list.apply(
    lambda r: f"สาขา {r['store_id']} ({r['province']}) ควรเติม {r['product_id']} "
              f"({r['product_taxonomies']}) จำนวน {r['reorder_qty']} ชิ้น "
              f"[โอกาสรายได้: {r['revenue_opportunity']:,.0f} THB] "
              f"[ความเสี่ยง Stockout: {r['stockout_risk']}]", axis=1
)

print("="*80)
print("  📌 ACTION LIST — SME Retail Reorder Recommendations")
print("="*80)
for i, row in action_list.iterrows():
    print(f"  {i+1:2d}. {row['action']}")

print("\n" + "="*80)

# ── Promotion Recommendations ─────────────────────────────────────────────────
good_promos = uplift_df[uplift_df['roi'] > 1.0].sort_values('roi', ascending=False)
bad_promos  = uplift_df[uplift_df['roi'] <= 1.0]

print("\n  📌 PROMOTION RECOMMENDATION")
print("="*80)
print(f"  ✅ Continue promotions (ROI > 1): {len(good_promos)} promo-product pairs")
print(f"  ❌ Revise promotions (ROI ≤ 1):   {len(bad_promos)} promo-product pairs")


In [ ]:

# ── Executive Summary ─────────────────────────────────────────────────────────
total_revenue = (sales_transaction['qty'] * sales_transaction['price']).sum()
total_lines   = len(sales_transaction)
total_units   = sales_transaction['qty'].sum()
reorder_oppt  = recommendations['revenue_opportunity'].sum()
high_risk_cnt = (recommendations['stockout_risk']=='HIGH').sum()
promo_roi_avg = uplift_df['roi'].mean()

summary = (
    "\n" + "="*76 + "\n" +
    "  EXECUTIVE SUMMARY -- SME Retail Revenue Intelligence\n" +
    f"  Data Science Solution | {datetime.now().strftime('%Y-%m-%d')}\n" +
    "="*76 + "\n" +
    "  BUSINESS CONTEXT\n" +
    "  * 3 stores (Chiang Mai, Bangkok, Phuket), 2 warehouses\n" +
    "  * 50 products across 5 categories, 500 customers, 7 promotions\n" +
    "="*76 + "\n" +
    "  2025 PERFORMANCE (MOCK DATA)\n" +
    f"  * Total Revenue      : {total_revenue:>15,.2f} THB\n" +
    f"  * Total Transactions : {total_lines:>15,}\n" +
    f"  * Total Units Sold   : {total_units:>15,}\n" +
    "="*76 + "\n" +
    "  FORECASTING MODEL\n" +
    f"  * Algorithm          : {selected_model_name}\n" +
    f"  * Baseline WAPE      : {baseline_wape:.4f}\n" +
    f"  * Model WAPE         : {selected_wape:.4f}\n" +
    f"  * Model Status       : {model_status}\n" +
    f"  * WAPE Improvement   : {improvement:.4f} ({improvement_pct:.1f}% vs baseline)\n" +
    "="*76 + "\n" +
    "  INVENTORY OPPORTUNITIES\n" +
    f"  * Reorder candidates : {len(recommendations)} store-product pairs\n" +
    f"  * Revenue Opportunity: {reorder_oppt:>15,.2f} THB\n" +
    f"  * HIGH Stockout Risk : {high_risk_cnt} items requiring immediate action\n" +
    "="*76 + "\n" +
    "  PROMOTION PERFORMANCE\n" +
    f"  * Avg Promotion ROI  : {promo_roi_avg:.3f}\n" +
    f"  * Profitable promos  : {len(good_promos)} pairs (ROI > 1.0)\n" +
    f"  * Unprofitable promos: {len(bad_promos)} pairs (ROI <= 1.0) -- revise discount\n" +
    "="*76 + "\n" +
    "  RECOMMENDATIONS\n" +
    f"  1. Immediate: Restock {high_risk_cnt} HIGH-risk SKUs to prevent revenue loss\n" +
    "  2. Short-term: Replace unprofitable discounts with targeted campaigns\n" +
    "  3. Medium-term: Automate weekly reorder via ML pipeline\n" +
    "  4. Long-term: Expand to customer-level CLV model for CRM\n" +
    "="*76
)
print(summary)

# Save summary
with open('data/executive_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("✅ All outputs saved to data/")


---
## 🗺️ End-to-End Pipeline Recap

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   SME Retail DS Pipeline                                │
├──────────────┬──────────────────────────────────────────────────────────┤
│  STEP        │  OUTPUT                                                  │
├──────────────┼──────────────────────────────────────────────────────────┤
│ 1. Mock Data │  8 CSV tables (50 products, 3 stores, 1 year)           │
│ 2. Validate  │  Null/dup/date-logic checks, expiry risk count          │
│ 3. EDA       │  Revenue trends, category split, weekday pattern        │
│ 4. Features  │  Lag/rolling, calendar, promo flags, encoded cats       │
│ 5. Forecast  │  GBM vs baseline WAPE: 0.42 vs 0.44                    │
│ 6. Inventory │  Reorder qty per store-product + revenue opportunity    │
│ 7. Promotion │  ROI per promo-product pair, profitable vs not          │
│ 8. MLOps     │  Weekly WAPE log, drift monitor, model registry JSON    │
│ 9. Output    │  Action list CSV + Executive Summary TXT                │
└──────────────┴──────────────────────────────────────────────────────────┘
```

### Output Files
| File | Description |
|------|-------------|
| `data/sales_transaction.csv` | Main POS fact table |
| `data/model_recommendation_output.csv` | **Priority reorder action list** |
| `data/executive_summary.txt` | Management summary |
| `mlops/model_registry.json` | Model versioning & metrics |
| `eda_overview.png` | EDA dashboard |
| `forecast_results.png` | Forecast accuracy charts |
| `inventory_dashboard.png` | Stockout & opportunity |
| `promotion_analysis.png` | Promotion ROI analysis |
| `mlops_monitoring.png` | Drift & WAPE tracking |
